In [ ]:
# CELL 0 — Environment setup for 02_motif_pipeline.ipynb
# Run ONCE, restart the Colab runtime, then run Cells 1-6.
# Prerequisite: 01_graph_pipeline.ipynb Cell 6 must have saved
#   temporal_edges.parquet to OUTPUT_DIR on Google Drive.

# 1. Install RAPIDS cuDF for T4 GPU (no-op if already installed)
try:
    import cudf  # noqa: F401
    print('cuDF already available.')
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           'cudf-cu12', '--extra-index-url', 'https://pypi.nvidia.com'])
    print('cuDF installed. RESTART the Colab runtime now, then re-run all cells.')

# 2. Mount Google Drive (idempotent — skipped if already mounted)
import os
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')
else:
    print('Drive already mounted.')

In [ ]:
from dataclasses import dataclass, field
from typing import List


@dataclass
class MotifConfig:
    """
    Hyperparameters for temporal motif mining.

    Temporal constraints
    --------------------
    delta : int
        Maximum step gap between two consecutive transactions in a motif.
        AML relay chains typically close within a few days; default 3 is
        intentionally tight. Widen only if dataset step != 1 day.

    Amount ratio constraints
    ------------------------
    rho_min, rho_max : float
        Allowed ratio a(i+1) / a(i) for consecutive transactions.
        Layering keeps amounts close to 1.0.
        Structuring splits downward (ratio < 1).
        [0.5, 2.0] is a starting range; tighten toward [0.7, 1.4] for
        capital-preservation behavior typical of layering.

    Minimum support (repetition)
    ----------------------------
    r_min_fanin  : minimum number of incoming sources for a fan-in node.
    r_min_fanout : minimum number of outgoing targets for a fan-out node.
    r_min_cycle  : minimum occurrences of a cycle before it is flagged.
    r_min_relay  : minimum occurrences of a relay chain.
    r_min_split_merge : minimum occurrences of a split-merge path.

    Statistical filtering
    ---------------------
    n_permutations : shuffles for null-model z-score estimation.
        30 is sufficient for early screening; raise to 100+ for final runs.
    z_min : z-score threshold below which a motif is discarded.

    Search limits
    -------------
    max_nodes : maximum nodes in a single motif instance (prune guard).
    max_edges : maximum edges in a single motif instance (prune guard).
    max_instances : hard cap on total matched instances kept in RAM.
        Prevents OOM on large windows; truncated types emit a warning.
        Set to None to disable.

    Window sizes
    ------------
    window_sizes : step counts for windowed search, aligned with graph module.
        [7, 14, 30] matches graph_schema §6.3 and community_spec §6.
    """

    # -- Temporal -------------------------------------------------------------
    delta: int = 3

    # -- Amount ratio ---------------------------------------------------------
    rho_min: float = 0.5
    rho_max: float = 2.0

    # -- Support thresholds (per motif type) ----------------------------------
    r_min_fanin:      int = 3   # at least 3 distinct sources into one node
    r_min_fanout:     int = 3   # at least 3 distinct targets from one node
    r_min_cycle:      int = 1   # a single observed cycle is already a signal
    r_min_relay:      int = 1
    r_min_split_merge: int = 1

    # -- Statistical filtering ------------------------------------------------
    n_permutations: int = 30
    z_min: float = 2.0

    # -- Search limits (prune guard) ------------------------------------------
    max_nodes: int = 4
    max_edges: int = 5
    # max_instances: total matched instances cap across all five matchers.
    # Prevents unbounded list growth on dense windows. 0 = disabled.
    max_instances: int = 100_000

    # -- Window sizes (aligned with GraphConfig.window_sizes) -----------------
    window_sizes: List[int] = field(default_factory=lambda: [7, 14, 30])


__all__ = ["MotifConfig"]

In [ ]:
"""
index.py — Event-level search indexes for motif matching.

Builds lightweight Python dict indexes from a window of transactions.
Matchers use these indexes instead of scanning the full DataFrame repeatedly.

Indexes produced by build_event_indexes():
    out_index  : {src_node -> [event_dict, ...]} sorted by step ascending
    in_index   : {dst_node -> [event_dict, ...]} sorted by step ascending
    step_index : {step     -> [event_dict, ...]}

Helper:
    edges_after_step(out_index, node, step) — forward-only edge lookup.
    filter_window(event_df, step_start, step_end) — slice a time window.

Guarantees:
- Time      : all buckets are sorted by step; edges_after_step enforces
              forward-only search (no backward traversal).
- Direction : out_index keys on src; in_index keys on dst; never merged.
- Memory    : plain Python dicts after indexing (no DataFrame held);
              gc.collect() called after build.
"""

from __future__ import annotations

import gc
from bisect import bisect_right
from collections import defaultdict
from typing import Dict, List

import pandas as pd
import os
# ---------------------------------------------------------------------------
# Type alias for an event record stored in the index
# ---------------------------------------------------------------------------

# Each event is a plain dict for zero-overhead lookup in matcher loops.
# Keys: event_id, step, src, dst, amount, is_sar
EventDict = Dict[str, object]


# ---------------------------------------------------------------------------
# Main index builder
# ---------------------------------------------------------------------------

def build_event_indexes(
    event_df: pd.DataFrame,
    src_col: str = "src_node",
    dst_col: str = "dst_node",
    step_col: str = "step",
    amount_col: str = "amount",
    alert_col: str = "is_sar",
) -> tuple[dict, dict, dict]:
    """
    Build three search indexes from a window of transactions.

    Parameters
    ----------
    event_df : pd.DataFrame
        A single time window of transactions.  Must be sorted by step
        (loader.py and iter_windows guarantee this).
        Supported column name variants:
          - canonical: src_node, dst_node (from loader.py)
          - legacy:    nameOrig, nameDest  (raw AMLGentex)
          - short:     src, dst           (old pipeline)
    src_col, dst_col, step_col, amount_col, alert_col : str
        Column name overrides.

    Returns
    -------
    out_index  : {src_node: [EventDict, ...]}  sorted by step ascending
    in_index   : {dst_node: [EventDict, ...]}  sorted by step ascending
    step_index : {step:     [EventDict, ...]}

    Notes
    -----
    - event_id is auto-assigned as the row index if absent (motif_spec §3.1).
    - itertuples() is used intentionally: the single-pass loop is O(n) and
      avoids creating three separate groupby-materialised DataFrames.
    - All three dicts are populated in one pass to minimise memory pressure.
    - Buckets are already in step order because event_df is pre-sorted.
    """
    # cuDF → pandas: motif index works on plain Python dicts after this point;
    # converting once here avoids per-row GPU tensor overhead in itertuples.

    if hasattr(event_df, "to_pandas"):
        event_df = event_df.to_pandas()

    # --- FIX WARNING 4: Defensive Integrity Guard ---
    # Validate monotonicity after pandas conversion but before indexing.
    if os.getenv("MOTIF_DEBUG") == "1":
        if not event_df[step_col].is_monotonic_increasing:
            raise ValueError(
                f"CRITICAL DATA ERROR: event_df must be sorted by '{step_col}' before indexing. "
                "Unsorted data will cause binary search (bisect) to return silent failures."
            )



    df = _normalize_columns(event_df, src_col, dst_col, step_col, amount_col, alert_col)
    df = _ensure_event_id(df)
    _validate_required(df)

    has_alert = "is_sar" in df.columns
    out_index:  dict = defaultdict(list)
    in_index:   dict = defaultdict(list)
    step_index: dict = defaultdict(list)

    # Single O(n) pass — itertuples permitted per coding conventions when
    # building an index that cannot be expressed as a vectorized operation.
    for row in df.itertuples(index=False):
        e: EventDict = {
            "event_id": int(row.event_id),
            "step":     int(row.step),
            "src":      int(row.src_node),
            "dst":      int(row.dst_node),
            "amount":   float(row.amount),
            "is_sar":   int(row.is_sar) if has_alert else 0,
        }
        out_index[e["src"]].append(e)
        in_index[e["dst"]].append(e)
        step_index[e["step"]].append(e)

    # Freeze to regular dicts: prevents accidental key creation on miss
    out_index  = dict(out_index)
    in_index   = dict(in_index)
    step_index = dict(step_index)
    # At the end of build_event_indexes, after populating out_index:
    out_steps = {node: [e["step"] for e in edges]
             for node, edges in out_index.items()}
    gc.collect()
    return out_index, in_index, step_index, out_steps


# ---------------------------------------------------------------------------
# Forward-only lookup helper
# ---------------------------------------------------------------------------


def edges_after_step(
    out_index: dict,
    node: int,
    step: int,
    out_steps: dict,
) -> List[EventDict]:
    """
    Return all outgoing edges from `node` with step > `step`.

    Used by matchers to expand a relay chain forward in time only.
    Binary search on the pre-sorted bucket avoids a linear scan.

    Parameters
    ----------
    out_index : dict
        Output of build_event_indexes().
    node : int
        Source node ID.
    step : int
        All returned edges have step strictly greater than this value.

    Returns
    -------
    List of EventDict, empty if node has no outgoing edges after step.
    """
    bucket = out_index.get(node)
    if not bucket:
        return []
    idx = bisect_right(out_steps[node], step)
    return bucket[idx:]



# ---------------------------------------------------------------------------
# Window slice helper
# ---------------------------------------------------------------------------

def filter_window(
    event_df: pd.DataFrame,
    step_start: int,
    step_end: int,
) -> pd.DataFrame:
    """
    Slice event_df to [step_start, step_end] (inclusive).

    Use before build_event_indexes() when working from a full loaded
    DataFrame rather than through iter_windows().

    Parameters
    ----------
    event_df : pd.DataFrame
        Must have a `step` column.
    step_start, step_end : int
        Inclusive step bounds.

    Returns
    -------
    Filtered DataFrame with reset index.  Direction and step order preserved.
    """
    mask = (event_df["step"] >= step_start) & (event_df["step"] <= step_end)
    return event_df[mask].reset_index(drop=True)


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _normalize_columns(
    df: pd.DataFrame,
    src_col: str,
    dst_col: str,
    step_col: str,
    amount_col: str,
    alert_col: str,
) -> pd.DataFrame:
    """
    Rename columns to canonical names (src_node, dst_node, step, amount, is_sar).
    Supports three naming variants without copying the full DataFrame.
    """
    rename: dict = {}

    # Caller-specified override names
    if src_col != "src_node" and src_col in df.columns:
        rename[src_col] = "src_node"
    if dst_col != "dst_node" and dst_col in df.columns:
        rename[dst_col] = "dst_node"
    if step_col != "step" and step_col in df.columns:
        rename[step_col] = "step"
    if amount_col != "amount" and amount_col in df.columns:
        rename[amount_col] = "amount"
    if alert_col != "is_sar" and alert_col in df.columns:
        rename[alert_col] = "is_sar"

    # Legacy / raw AMLGentex column names
    cols = set(df.columns)
    if "src_node" not in cols and "nameOrig" in cols:
        rename["nameOrig"] = "src_node"
    if "dst_node" not in cols and "nameDest" in cols:
        rename["nameDest"] = "dst_node"
    # Short-form names (old pipeline)
    if "src_node" not in cols and "src" in cols:
        rename["src"] = "src_node"
    if "dst_node" not in cols and "dst" in cols:
        rename["dst"] = "dst_node"
    # is_sar / Is Laundering / is_laundering aliases
    if "is_sar" not in cols:
        for alias in ("is_laundering", "Is Laundering", "isSAR"):
            if alias in cols:
                rename[alias] = "is_sar"
                break

    if rename:
        df = df.rename(columns=rename)
    return df


def _ensure_event_id(df: pd.DataFrame) -> pd.DataFrame:
    """Auto-assign event_id from row index if the column is absent."""
    if "event_id" not in df.columns:
        df = df.copy()
        df["event_id"] = range(len(df))
    return df


def _validate_required(df: pd.DataFrame) -> None:
    """Raise ValueError if mandatory columns are missing."""
    required = {"event_id", "step", "src_node", "dst_node", "amount"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"event_df is missing required columns: {sorted(missing)}. "
            f"Available columns: {sorted(df.columns)}"
        )


__all__ = [
    "build_event_indexes",
    "edges_after_step",
    "filter_window",
]

In [ ]:
"""
matchers.py — Temporal motif matching for AML detection.

Five templates:
    fan_in      : many -> one within delta steps
    fan_out     : one -> many within delta steps
    cycle_3     : u -> v -> w -> u with strict time order
    relay_4     : u -> v -> w -> x with strict time order
    split_merge : u -> v1 -> z and u -> v2 -> z

General rules for every matcher (guide §4):
    1. Forward-only — step strictly increases at each hop.
    2. Early stop — prune immediately when any constraint fails.
    3. No side effects — indexes are never modified.
    4. Return list[dict] — each dict is one motif instance.

Instance dict keys (guide §8):
    motif_type, nodes, edges (event_id list),
    steps, amounts, lags, ratios, n_alert

No pandas / cuDF imports — only plain Python dicts and lists.
"""

from __future__ import annotations

# ---------------------------------------------------------------------------
# Primitive constraint helpers
# ---------------------------------------------------------------------------

def _ratio_ok(a_prev: float, a_curr: float, rho_min: float, rho_max: float) -> bool:
    """True if a_curr / a_prev is in [rho_min, rho_max]."""
    if a_prev <= 0:
        return False
    r = a_curr / a_prev
    return rho_min <= r <= rho_max


def _lag_ok(step_prev: int, step_curr: int, delta: int) -> bool:
    """True if 0 < step_curr - step_prev <= delta."""
    lag = step_curr - step_prev
    return 0 < lag <= delta


# ---------------------------------------------------------------------------
# Instance constructor
# ---------------------------------------------------------------------------

def _make_instance(
    motif_type: str,
    edges: list[dict],
    nodes: list[int],
) -> dict:
    """
    Build a standardised motif instance dict from an ordered edge list.

    `edges` must be in chronological order (step ascending).
    """
    steps   = [e["step"]   for e in edges]
    amounts = [e["amount"] for e in edges]
    lags    = [steps[i] - steps[i - 1] for i in range(1, len(steps))]
    ratios  = [
        round(amounts[i] / amounts[i - 1], 4) if amounts[i - 1] > 0 else 0.0
        for i in range(1, len(amounts))
    ]
    return {
        "motif_type": motif_type,
        "nodes":      nodes,
        "edges":      [e["event_id"] for e in edges],
        "steps":      steps,
        "amounts":    amounts,
        "lags":       lags,
        "ratios":     ratios,
        "n_alert":    sum(e.get("is_sar", 0) for e in edges),
    }


# ---------------------------------------------------------------------------
# Fan-in  — FIX R3: add seen-set dedup
# ---------------------------------------------------------------------------

def find_fanin(
    in_index: dict,
    cfg: MotifConfig,
) -> list[dict]:
    """
    Fan-in: r_min_fanin distinct sources -> same destination x, within delta steps.

        u1 -> x
        u2 -> x   (step_u2 - step_u1 <= delta)
        u3 -> x   ...

    Constraints:
        - All sources distinct.
        - All arrivals in [t0, t0 + delta].
        - Amount ratio of each edge vs seed in [rho_min, rho_max].
        - At least r_min_fanin sources found.

    FIX R3: A seen-set keyed by (frozenset(source_ids), destination) prevents
    the O(n^2) duplicate groups produced by seed-shifting without dedup.
    """
    results = []
    # seen: prevents re-emitting the same group under a different seed edge.
    seen: set = set()

    for x, incoming in in_index.items():
        n = len(incoming)
        if n < cfg.r_min_fanin:
            continue    # prune: not enough edges to form a group

        for i in range(n):
            seed   = incoming[i]
            t0     = seed["step"]
            a_prev = seed["amount"]
            seen_src = {seed["src"]}
            group  = [seed]

            for j in range(i + 1, n):
                e = incoming[j]
                # Prune: window exceeded — bucket is sorted, so break early
                if e["step"] - t0 > cfg.delta:
                    break
                # Prune: duplicate source — skip, do not break
                if e["src"] in seen_src:
                    continue
                # Prune: amount ratio
                if not _ratio_ok(a_prev, e["amount"], cfg.rho_min, cfg.rho_max):
                    continue

                seen_src.add(e["src"])
                group.append(e)
                a_prev = e["amount"]

            if len(group) >= cfg.r_min_fanin:
                # Canonical key: frozenset of participating source IDs + destination.
                # This collapses all seed permutations of the same source group.
                key = (frozenset(e["src"] for e in group), x)
                if key in seen:
                    continue        # already emitted this exact group
                seen.add(key)

                nodes = [e["src"] for e in group] + [x]
                results.append(_make_instance("fanin", group, nodes))

    return results


# ---------------------------------------------------------------------------
# Fan-out  — unchanged; correct as-is
# ---------------------------------------------------------------------------

def find_fanout(
    out_index: dict,
    cfg: MotifConfig,
    out_steps: dict,
) -> list[dict]:
    """
    Fan-out: source x -> r_min_fanout distinct destinations, within delta steps.

        x -> v1
        x -> v2   (step_v2 - step_v1 <= delta)
        x -> v3   ...

    Constraints mirror find_fanin (symmetric).
    Returns list of instance dicts.
    """
    results = []

    for x, outgoing in out_index.items():
        n = len(outgoing)
        if n < cfg.r_min_fanout:
            continue

        for i in range(n):
            seed   = outgoing[i]
            t0     = seed["step"]
            a0     = seed["amount"]
            seen   = {seed["dst"]}
            group  = [seed]
            a_prev = a0

            for j in range(i + 1, n):
                e = outgoing[j]

                if e["step"] - t0 > cfg.delta:
                    break

                if e["dst"] in seen:
                    continue

                if not _ratio_ok(a_prev, e["amount"], cfg.rho_min, cfg.rho_max):
                    continue

                seen.add(e["dst"])
                group.append(e)
                a_prev = e["amount"]

            if len(group) >= cfg.r_min_fanout:
                nodes = [x] + [e["dst"] for e in group]
                results.append(_make_instance("fanout", group, nodes))

    return results


# ---------------------------------------------------------------------------
# Cycle-3  — unchanged; has seen-set dedup
# ---------------------------------------------------------------------------

def find_cycle3(
    out_index: dict,
    cfg: MotifConfig,
    out_steps: dict,
) -> list[dict]:
    """
    Cycle-3: u -> v -> w -> u with strictly increasing steps.

    Constraints:
        - step_e1 < step_e2 < step_e3 (strict)
        - Each consecutive lag <= delta
        - Amount ratio at each hop in [rho_min, rho_max]
        - u, v, w are three distinct nodes

    Uses edges_after_step() for forward-only bucket access.
    Returns list of instance dicts.
    """
    results = []
    seen = set()
    for u, edges_u in out_index.items():
        for e1 in edges_u:
            v  = e1["dst"]
            t1 = e1["step"]
            a1 = e1["amount"]

            if v == u:
                continue

            # e2: v -> w, step in (t1, t1 + delta]
            for e2 in edges_after_step(out_index, v, t1, out_steps):
                if e2["step"] - t1 > cfg.delta:
                    break   # bucket sorted -> nothing after is valid

                w  = e2["dst"]
                a2 = e2["amount"]

                if w == u or w == v:
                    continue

                if not _ratio_ok(a1, a2, cfg.rho_min, cfg.rho_max):
                    continue

                # e3: w -> u, step in (t2, t2 + delta]
                for e3 in edges_after_step(out_index, w, e2["step"], out_steps):
                    if e3["step"] - e2["step"] > cfg.delta:
                        break

                    if e3["dst"] != u:
                        continue

                    if not _ratio_ok(a2, e3["amount"], cfg.rho_min, cfg.rho_max):
                        continue
                    key = frozenset([e1["event_id"],
                                     e2["event_id"],
                                     e3["event_id"]])
                    if key in seen:
                        continue
                    seen.add(key)

                    results.append(_make_instance(
                        "cycle3",
                        [e1, e2, e3],
                        [u, v, w, u],
                    ))

    return results


# ---------------------------------------------------------------------------
# Relay-4  — unchanged; has seen-set dedup
# ---------------------------------------------------------------------------

def find_relay4(
    out_index: dict,
    cfg: MotifConfig,
    out_steps: dict,
) -> list[dict]:
    """
    Relay-4: u -> v -> w -> x with strictly increasing steps.

    Constraints:
        - step_e1 < step_e2 < step_e3 (strict)
        - Each consecutive lag <= delta
        - Amount ratio at each hop in [rho_min, rho_max]
        - u, v, w, x are four distinct nodes

    Uses edges_after_step() for forward-only access.
    Returns list of instance dicts.
    """
    results = []
    seen = set()
    for u, edges_u in out_index.items():
        for e1 in edges_u:
            v  = e1["dst"]
            t1 = e1["step"]
            a1 = e1["amount"]

            if v == u:
                continue

            for e2 in edges_after_step(out_index, v, t1, out_steps):
                if e2["step"] - t1 > cfg.delta:
                    break

                w  = e2["dst"]
                a2 = e2["amount"]

                if w in (u, v):
                    continue

                if not _ratio_ok(a1, a2, cfg.rho_min, cfg.rho_max):
                    continue

                for e3 in edges_after_step(out_index, w, e2["step"], out_steps):
                    if e3["step"] - e2["step"] > cfg.delta:
                        break

                    x  = e3["dst"]
                    a3 = e3["amount"]

                    if x in (u, v, w):
                        continue

                    if not _ratio_ok(a2, a3, cfg.rho_min, cfg.rho_max):
                        continue

                    key = frozenset([e1["event_id"],
                                     e2["event_id"],
                                     e3["event_id"]])
                    if key in seen:
                        continue
                    seen.add(key)

                    results.append(_make_instance(
                        "relay4",
                        [e1, e2, e3],
                        [u, v, w, x],
                    ))

    return results


# ---------------------------------------------------------------------------
# Split-merge  — FIX R2: single-best-edge + dedup + remove dead code
# ---------------------------------------------------------------------------

def find_split_merge(
    out_index: dict,
    in_index: dict,
    cfg: MotifConfig,
    out_steps: dict,
) -> list[dict]:
    """
    Split-merge: source splits to two intermediaries that recombine at one target.

        u -> v1 -> z
        u -> v2 -> z

    Two-phase:
        Phase 1: find split pairs (u -> v1, u -> v2) within delta steps.
        Phase 2: for each pair, find a common target z that both v1 and v2
                 reach within 2 * delta steps of the split start.

    Constraints:
        - u, v1, v2, z are four distinct nodes
        - Amount ratio at every hop in [rho_min, rho_max]
        - All events in chronological order per hop

    FIX R2a — single-best-edge: v1_targets keeps only the FIRST (earliest)
        edge from v1 to each z, not all edges.  This avoids the Cartesian
        product explosion in the original code where v1_targets[z] was a list
        and every (e_v1z, e_v2z) combo was emitted as a separate instance.

    FIX R2b — seen-set: a frozenset over all four event_ids prevents the same
        (u, v1, v2, z) quad from being re-emitted via different traversal paths.

    FIX R2c — v1_targets cached per (u, v1): computed once before the v2 loop,
        not rebuilt for every (v1, v2) pair.
    """
    results = []
    seen: set = set()   # frozenset of 4 event_ids to deduplicate quads

    for u, outgoing_u in out_index.items():
        n = len(outgoing_u)

        for i in range(n):
            e_uv1 = outgoing_u[i]
            v1    = e_uv1["dst"]
            t0    = e_uv1["step"]
            a_uv1 = e_uv1["amount"]

            # FIX R2c — build v1_targets once per (u, v1) before the v2 loop.
            # Maps z -> the single earliest v1->z edge within 2*delta of t0.
            v1_targets: dict = {}
            for e in edges_after_step(out_index, v1, t0, out_steps):
                if e["step"] - t0 > 2 * cfg.delta:
                    break
                z = e["dst"]
                if z in (u, v1):
                    continue
                if not _ratio_ok(a_uv1, e["amount"], cfg.rho_min, cfg.rho_max):
                    continue
                # FIX R2a — keep only the first (earliest) edge to z.
                if z not in v1_targets:
                    v1_targets[z] = e

            # Skip this v1 entirely if it reaches no valid targets
            if not v1_targets:
                continue

            # Phase 1: iterate v2 candidates (outgoing edges from u after e_uv1)
            for j in range(i + 1, n):
                e_uv2 = outgoing_u[j]

                # Prune: split window exceeded
                if e_uv2["step"] - t0 > cfg.delta:
                    break

                v2 = e_uv2["dst"]
                if v2 == v1 or v2 == u:
                    continue

                # Prune: split amount ratio
                if not _ratio_ok(a_uv1, e_uv2["amount"], cfg.rho_min, cfg.rho_max):
                    continue

                # Phase 2: find v2 -> z where z is already reachable from v1
                for e_v2z in edges_after_step(out_index, v2, t0, out_steps):
                    if e_v2z["step"] - t0 > 2 * cfg.delta:
                        break
                    z = e_v2z["dst"]

                    # z must be in v1_targets and must be a new node
                    if z not in v1_targets or z in (u, v1, v2):
                        continue

                    # Amount ratio on the v2->z leg
                    if not _ratio_ok(e_uv2["amount"], e_v2z["amount"],
                                     cfg.rho_min, cfg.rho_max):
                        continue

                    e_v1z = v1_targets[z]   # single best edge

                    # FIX R2b — dedup by the four event_ids
                    key = frozenset([e_uv1["event_id"], e_uv2["event_id"],
                                     e_v1z["event_id"], e_v2z["event_id"]])
                    if key in seen:
                        continue
                    seen.add(key)

                    all_edges = sorted(
                        [e_uv1, e_uv2, e_v1z, e_v2z],
                        key=lambda e: e["step"],
                    )
                    results.append(
                        _make_instance("split_merge", all_edges, [u, v1, v2, z])
                    )

    return results


# ---------------------------------------------------------------------------
# Convenience: run all matchers on a pre-built index set
# FIX R1: max_instances cap prevents unbounded RAM accumulation.
# ---------------------------------------------------------------------------

def run_all_matchers(
    out_index: dict,
    in_index: dict,
    cfg: MotifConfig,
    out_steps: dict[int, list[dict]],
) -> list[dict]:
    """
    Run all five matchers and return a combined instance list.

    Parameters
    ----------
    out_index, in_index : dict
        Output of build_event_indexes().
    cfg : MotifConfig
        Motif configuration.  cfg.max_instances caps the total list size
        (0 = disabled).

    Returns
    -------
    Combined list of all matched motif instances across all types.
    If cfg.max_instances > 0, each matcher's result is truncated to
    cfg.max_instances before concatenation, and a warning is printed.
    """
    # Run each matcher independently so we can apply the cap per type.
    matchers = {
        "fanin":       find_fanin(in_index, cfg),
        "fanout":      find_fanout(out_index, cfg, out_steps),
        "cycle3":      find_cycle3(out_index, cfg, out_steps),
        "relay4":      find_relay4(out_index, cfg, out_steps),
        "split_merge": find_split_merge(out_index, in_index, cfg, out_steps),
    }

    combined = []
    cap = cfg.max_instances if cfg.max_instances > 0 else None
    for mtype, instances in matchers.items():
        if cap and len(instances) > cap:
            # Warn and truncate; do NOT silently skip — the signal is still present.
            print(f"  [WARN] {mtype}: {len(instances):,} instances exceed "
                  f"max_instances={cap:,}; truncating to {cap:,}.")
            instances = instances[:cap]
        combined.extend(instances)

    return combined


__all__ = [
    "find_fanin",
    "find_fanout",
    "find_cycle3",
    "find_relay4",
    "find_split_merge",
    "run_all_matchers",
]

In [ ]:
# scoring.py — support counting, filtering, null-model z-score.
# FIX R4a: _run_matchers_on_df truncates instances per null pass (cap arg).
# FIX R4b: compute_null_zscore subsamples event_df per perm (sample_frac).
# FIX R4c: compute_null_zscore returns {} when observed_counts is empty.

from __future__ import annotations
import copy, gc
from collections import Counter
from typing import Dict, List
import numpy as np
import pandas as pd


# -- Support counting — unchanged ----------------------------------------

def count_support(motif_instances: List[dict]) -> Dict[str, int]:
    # Count matched instances by motif type. Returns {motif_type: count}.
    c: Counter = Counter()
    for inst in motif_instances:
        c[inst['motif_type']] += 1
    return dict(c)


# -- Filtering — unchanged logic -----------------------------------------

def filter_motifs(
    instances: List[dict],
    cfg: MotifConfig,
    zscore_results: Dict[str, dict] | None = None,
) -> List[dict]:
    # Apply per-type r_min and optional z_min threshold.
    # fan-in/fan-out r_min is enforced by the matchers; require >=1 here.
    r_min_map = {
        'fanin':       1,
        'fanout':      1,
        'cycle3':      cfg.r_min_cycle,
        'relay4':      cfg.r_min_relay,
        'split_merge': cfg.r_min_split_merge,
    }
    support = count_support(instances)
    keep_types = set()
    for mtype, count in support.items():
        if count < r_min_map.get(mtype, 1):
            continue
        if zscore_results is not None:
            if zscore_results.get(mtype, {}).get('zscore', 0.0) < cfg.z_min:
                continue
        keep_types.add(mtype)
    return [inst for inst in instances if inst['motif_type'] in keep_types]


# -- Null-model helpers --------------------------------------------------

def _shuffle_timestamps(event_df: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    # Shuffle step values per source node; re-sort globally.
    # Preserves degree + amount distributions; breaks sequential motif patterns.
    df_null = event_df.copy()
    for src, idx in event_df.groupby('src_node').groups.items():
        df_null.loc[idx, 'step'] = rng.permutation(event_df.loc[idx, 'step'].to_numpy())
    return df_null.sort_values(['step', 'event_id']).reset_index(drop=True)


def _run_matchers_on_df(
    event_df: pd.DataFrame,
    cfg: MotifConfig,
    cap: int = 0,
) -> Dict[str, int]:
    # FIX R4a: build indexes, run matchers with per-type cap, return counts.
    # cap overrides cfg.max_instances for null runs, bounding RAM per pass.
    out_idx, in_idx, _, out_steps = build_event_indexes(event_df)
    cfg_null = copy.copy(cfg)
    if cap > 0:
        cfg_null.max_instances = min(cap, cfg.max_instances) if cfg.max_instances > 0 else cap
    instances = run_all_matchers(out_idx, in_idx, cfg_null, out_steps)
    counts = count_support(instances)
    del out_idx, in_idx, instances
    return counts


def compute_null_zscore(
    observed_counts: Dict[str, int],
    event_df: pd.DataFrame,
    cfg: MotifConfig,
    seed: int = 42,
    verbose: bool = False,
    sample_frac: float = 1.0,
    null_cap: int = 10_000,
) -> Dict[str, dict]:
    # Compute z(M) = (C_obs - mean_null) / (std_null + eps) per motif type.
    # FIX R4c: return {} immediately if nothing observed (skip all perms).
    # FIX R4b: sample_frac < 1.0 subsamples event_df per perm to cut RAM.
    # FIX R4a: null_cap forwarded to _run_matchers_on_df per perm.
    #
    # sample_frac : float in (0,1] — fraction of rows per permutation.
    #   Use 0.2-0.5 for large windows; 1.0 = no sampling (original).
    # null_cap : int — per-type instance cap inside each null pass.

    # FIX R4c
    if not observed_counts:
        return {}

    null_counts: Dict[str, list] = {mt: [] for mt in observed_counts}
    rng = np.random.default_rng(seed)

    for i in range(cfg.n_permutations):
        if verbose:
            print(f'  Null permutation {i + 1}/{cfg.n_permutations}...')

        # FIX R4b — optional row sampling
        if sample_frac < 1.0:
            df_perm = (
                event_df
                .sample(frac=sample_frac, random_state=int(rng.integers(1_000_000)))
                .sort_values(['step', 'event_id'])
                .reset_index(drop=True)
            )
        else:
            df_perm = event_df

        df_null = _shuffle_timestamps(df_perm, rng)
        perm_counts = _run_matchers_on_df(df_null, cfg, cap=null_cap)
        for mt in observed_counts:
            null_counts[mt].append(perm_counts.get(mt, 0))
        del df_null, perm_counts
        if sample_frac < 1.0:
            del df_perm
        gc.collect()

    results: Dict[str, dict] = {}
    for mt, c_obs in observed_counts.items():
        arr       = np.array(null_counts[mt], dtype=float)
        mean_null = float(arr.mean())
        std_null  = float(arr.std())
        zscore    = (c_obs - mean_null) / (std_null + 1e-9)
        results[mt] = {
            'observed':  c_obs,
            'mean_null': round(mean_null, 2),
            'std_null':  round(std_null, 2),
            'zscore':    round(zscore, 3),
        }
    return results


__all__ = ['count_support', 'filter_motifs', 'compute_null_zscore']

In [ ]:
"""
features.py — Motif feature extraction for ML downstream use.

Two primary feature tables:
    1. Entity-level (per node, per motif type)
       Output: build_entity_motif_features()  → long format
               build_entity_feature_wide()     → wide format (one row per node)

    2. Window-level (per time bucket, per motif type)
       Output: build_window_motif_features()

Export:
    save_features() — writes parquet to a configurable path (Google Drive).

Feature columns produced (motif_spec §7 / guide §8):
    count             — raw instance count
    avg_amount        — mean amount across all edges in matched instances
    avg_ratio         — mean amount preservation ratio across hops
    ratio_std         — std of amount ratio (higher = more irregular)
    avg_lag           — mean step gap between consecutive hops
    max_lag           — worst-case lag (flags delayed relay)
    avg_n_alert_edges — mean flagged edges per instance
    zscore            — significance vs. null model (from compute_null_zscore)
    freq_by_degree    — count / node degree (passed in by caller)
    freq_by_volume    — count / total transaction volume in window

Guarantees:
    - Time   : window_start assigned from min(steps) of each instance.
    - Direction: node role (src/dst) preserved in entity table; not collapsed.
    - Memory : row-building is O(instances × nodes_per_instance), not O(all transactions).
               groupby aggregation is vectorized. No unnecessary intermediate copies.
"""

from __future__ import annotations

import os
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Entity-level features
# ---------------------------------------------------------------------------

def build_entity_motif_features(
    motif_instances: List[dict],
    zscore_table: Optional[Dict[str, dict]] = None,
    node_degree: Optional[Dict[int, int]] = None,
    total_volume: float = 0.0,
) -> pd.DataFrame:
    """
    Aggregate motif instances into a feature table per (node, motif_type).

    For each node participating in at least one motif instance, computes:
        count             — number of instances the node appears in
        avg_amount        — mean of per-instance mean amounts
        avg_ratio         — mean amount preservation ratio across hops
        ratio_std         — std of ratio sequence (instability signal)
        avg_lag           — mean hop lag
        max_lag           — maximum hop lag seen
        avg_n_alert_edges — mean SAR-flagged edges per instance
        zscore            — motif significance (from compute_null_zscore)
        freq_by_degree    — count / node out-degree (0 if degree unknown)
        freq_by_volume    — count / total_volume (0 if volume == 0)

    Parameters
    ----------
    motif_instances : list[dict]
        Combined output of run_all_matchers() or filter_motifs().
    zscore_table : dict, optional
        Output of compute_null_zscore(). Keys are motif_type strings.
    node_degree : dict, optional
        {node_id: degree} mapping. Used to compute freq_by_degree.
        If None, freq_by_degree is set to 0 for all rows.
    total_volume : float
        Total transaction amount in the current window.
        Used to compute freq_by_volume. Set to 0 to skip.

    Returns
    -------
    pd.DataFrame with columns:
        node, motif_type, count, avg_amount, avg_ratio, ratio_std,
        avg_lag, max_lag, avg_n_alert_edges, zscore,
        freq_by_degree, freq_by_volume
    One row per (node, motif_type). Sorted by node, motif_type.
    """
    _EMPTY_COLS = [
        "node", "motif_type", "count",
        "avg_amount", "avg_ratio", "ratio_std",
        "avg_lag", "max_lag", "avg_n_alert_edges",
        "zscore", "freq_by_degree", "freq_by_volume",
    ]

    if not motif_instances:
        return pd.DataFrame(columns=_EMPTY_COLS)

    rows = []
    for inst in motif_instances:
        mt      = inst["motif_type"]
        amounts = inst.get("amounts", [])
        lags    = inst.get("lags", [])
        ratios  = inst.get("ratios", [])
        n_alert = inst.get("n_alert", 0)
        zs      = zscore_table[mt]["zscore"] if (zscore_table and mt in zscore_table) else np.nan

        avg_amt   = float(np.mean(amounts))   if amounts else 0.0
        avg_ratio = float(np.mean(ratios))    if ratios  else np.nan
        r_std     = float(np.std(ratios))     if ratios  else np.nan
        avg_lag   = float(np.mean(lags))      if lags    else 0.0
        max_lag   = float(max(lags))          if lags    else 0.0

        for node in set(inst.get("nodes", [])):
            rows.append({
                "node":             int(node),
                "motif_type":       mt,
                "avg_amount":       avg_amt,
                "avg_ratio":        avg_ratio,
                "ratio_std":        r_std,
                "avg_lag":          avg_lag,
                "max_lag":          max_lag,
                "n_alert_edges":    n_alert,
                "zscore":           zs,
            })

    if not rows:
        return pd.DataFrame(columns=_EMPTY_COLS)

    df = pd.DataFrame(rows)

    agg = (
        df.groupby(["node", "motif_type"], as_index=False)
        .agg(
            count               =("avg_amount",     "count"),
            avg_amount          =("avg_amount",      "mean"),
            avg_ratio           =("avg_ratio",       "mean"),
            ratio_std           =("ratio_std",       "mean"),
            avg_lag             =("avg_lag",         "mean"),
            max_lag             =("max_lag",         "max"),
            avg_n_alert_edges   =("n_alert_edges",   "mean"),
            zscore              =("zscore",          "first"),
        )
    )

    # Normalised frequency by node degree
    if node_degree:
        agg["freq_by_degree"] = agg.apply(
            lambda r: r["count"] / node_degree.get(int(r["node"]), 1),
            axis=1,
        )
    else:
        agg["freq_by_degree"] = 0.0

    # Normalised frequency by window transaction volume
    if total_volume > 0:
        agg["freq_by_volume"] = agg["count"] / total_volume
    else:
        agg["freq_by_volume"] = 0.0

    return agg.sort_values(["node", "motif_type"]).reset_index(drop=True)


# ---------------------------------------------------------------------------
# Wide-format pivot (one row per node, all motif types as columns)
# ---------------------------------------------------------------------------

def build_entity_feature_wide(entity_features: pd.DataFrame) -> pd.DataFrame:
    """
    Pivot entity_motif_features from long → wide format.

    Long:  [node, motif_type, count, avg_amount, ...]
    Wide:  [node, fanin_count, fanin_avg_amount, ..., cycle3_count, ...]

    Use when a single feature vector per node is needed for model input.
    Missing (node, motif_type) combinations are filled with 0.

    Parameters
    ----------
    entity_features : pd.DataFrame
        Output of build_entity_motif_features().

    Returns
    -------
    pd.DataFrame: one row per node.
    Column naming: {motif_type}_{metric}  e.g. fanin_count, relay4_avg_lag.
    """
    if entity_features.empty:
        return pd.DataFrame()

    metric_cols = [
        "count", "avg_amount", "avg_ratio", "ratio_std",
        "avg_lag", "max_lag", "avg_n_alert_edges", "zscore",
        "freq_by_degree", "freq_by_volume",
    ]
    available = [c for c in metric_cols if c in entity_features.columns]

    wide = entity_features.pivot_table(
        index="node",
        columns="motif_type",
        values=available,
        aggfunc="first",
    )

    # Flatten multi-level columns: (metric, motif_type) → "motif_type_metric"
    wide.columns = [f"{mt}_{metric}" for metric, mt in wide.columns]
    wide = wide.fillna(0).reset_index()

    return wide


# ---------------------------------------------------------------------------
# Window-level features
# ---------------------------------------------------------------------------

def build_window_motif_features(
    motif_instances: List[dict],
    window_size: int = 7,
) -> pd.DataFrame:
    """
    Aggregate motif instances into a feature table per (window_start, motif_type).

    For each time bucket, computes:
        count             — number of matched instances
        total_amount      — total transaction amount across all edges
        avg_lag           — mean hop lag across all instances
        n_alert_edges     — total SAR-flagged edges
        suspicious_ratio  — fraction of instances with at least one alert edge

    Parameters
    ----------
    motif_instances : list[dict]
        Output from run_all_matchers() or filter_motifs().
    window_size : int
        Number of steps per bucket. Default 7 = one week if step == 1 day.

    Returns
    -------
    pd.DataFrame with columns:
        window_start, motif_type, count, total_amount,
        avg_lag, n_alert_edges, suspicious_ratio
    Sorted by window_start, motif_type.
    """
    _EMPTY_COLS = [
        "window_start", "motif_type", "count",
        "total_amount", "avg_lag", "n_alert_edges", "suspicious_ratio",
    ]

    if not motif_instances:
        return pd.DataFrame(columns=_EMPTY_COLS)

    rows = []
    for inst in motif_instances:
        steps   = inst.get("steps", [])
        amounts = inst.get("amounts", [])
        lags    = inst.get("lags", [])
        n_alert = inst.get("n_alert", 0)

        if not steps:
            continue

        # Assign to bucket by first event's step — time-preserving
        window_start = (min(steps) // window_size) * window_size

        rows.append({
            "window_start": window_start,
            "motif_type":   inst["motif_type"],
            "total_amount": float(sum(amounts)) if amounts else 0.0,
            "avg_lag":      float(np.mean(lags)) if lags else 0.0,
            "n_alert_edges": n_alert,
            "has_alert":    int(n_alert > 0),
        })

    if not rows:
        return pd.DataFrame(columns=_EMPTY_COLS)

    df = pd.DataFrame(rows)

    agg = (
        df.groupby(["window_start", "motif_type"], as_index=False)
        .agg(
            count            =("total_amount",  "count"),
            total_amount     =("total_amount",  "sum"),
            avg_lag          =("avg_lag",        "mean"),
            n_alert_edges    =("n_alert_edges",  "sum"),
            suspicious_ratio =("has_alert",      "mean"),
        )
    )

    return agg.sort_values(["window_start", "motif_type"]).reset_index(drop=True)


# ---------------------------------------------------------------------------
# Export to parquet / CSV (configurable path for Google Drive)
# ---------------------------------------------------------------------------

# Default export directory — override in Colab:
#   import src.motif.features as mf
#   mf.DEFAULT_EXPORT_DIR = "/content/drive/MyDrive/aml_outputs"
DEFAULT_EXPORT_DIR: str = "outputs/motif_features"


def save_features(
    df: pd.DataFrame,
    filename: str,
    export_dir: Optional[str] = None,
    fmt: str = "parquet",
) -> str:
    """
    Save a feature DataFrame to disk (parquet or CSV).

    Parameters
    ----------
    df : pd.DataFrame
        Any feature table produced by this module.
    filename : str
        Base filename without extension, e.g. "entity_features_w30".
    export_dir : str, optional
        Target directory.  Defaults to DEFAULT_EXPORT_DIR.
        Set to "/content/drive/MyDrive/<your_path>" in Colab.
    fmt : str
        "parquet" (default, smaller) or "csv".

    Returns
    -------
    str : full path of the saved file.

    Notes
    -----
    - Parquet is preferred for downstream pandas / spark reads.
    - CSV is a fallback for manual inspection or Drive sharing.
    - The directory is created if it does not exist.
    """
    out_dir = Path(export_dir or DEFAULT_EXPORT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    ext  = "parquet" if fmt == "parquet" else "csv"
    path = out_dir / f"{filename}.{ext}"

    if fmt == "parquet":
        df.to_parquet(path, index=False)
    else:
        df.to_csv(path, index=False)

    print(f"  Saved {len(df):,} rows → {path}")
    return str(path)


__all__ = [
    "build_entity_motif_features",
    "build_entity_feature_wide",
    "build_window_motif_features",
    "save_features",
    "DEFAULT_EXPORT_DIR",
]

In [ ]:
# CELL 6 — Motif Pipeline: run end-to-end and save artifacts.
# Prerequisites:
#   - Cell 0 run and runtime restarted (RAPIDS installed, Drive mounted)
#   - Cells 1-5 executed (all functions defined in this session)
#   - 01_graph_pipeline.ipynb Cell 6 artifacts saved to OUTPUT_DIR on Drive

import os, gc, json
import numpy as np
import pandas as pd

# -- Drive guard: fail immediately if Drive not mounted -----------------
if not os.path.isdir('/content/drive/MyDrive'):
    raise RuntimeError(
        'Google Drive is not mounted. '
        'Run Cell 0: from google.colab import drive; drive.mount("/content/drive")'
    )

# -- USER CONFIG --------------------------------------------------------
OUTPUT_DIR       = '/content/drive/MyDrive/AML/outputs'  # <- same as graph Cell 6
MOTIF_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'motif')
RUN_ZSCORE  = True     # False = skip null model (faster for debugging)
N_PERMS     = None     # e.g. 10 to override cfg_motif.n_permutations
SAMPLE_FRAC = 1.0      # null model row sampling (0.2-0.5 for large windows)
NULL_CAP    = 10_000   # per-type instance cap per null permutation
# -----------------------------------------------------------------------

os.makedirs(MOTIF_OUTPUT_DIR, exist_ok=True)

# -- Step 1: Load temporal_edges from Drive -----------------------------
print('[1/6] Loading temporal_edges...')
te = pd.read_parquet(os.path.join(OUTPUT_DIR, 'temporal_edges.parquet'))
print(f'      shape: {te.shape}  |  columns: {list(te.columns)}')

# -- Step 2: Reconstruct event_df from relay pairs ----------------------
# temporal_edges is a self-join on the relay node: each row = (tx_A, tx_B).
# event_df = union(left side, right side), de-duplicated.
# NOTE (analysis C4): event_df is a strict subset of the original window.
#   Transactions with no relay hop within DELTA_W steps are absent.
#   Isolated source/sink nodes cannot form multi-hop motifs, so this is
#   acceptable; however, fan-in/fan-out at low-degree nodes may be
#   slightly underestimated.
print('\n[2/6] Reconstructing event_df...')
left  = te[['src_1','dst_1','step_1','amount_1','alert_1']].rename(columns={
    'src_1':'src_node', 'dst_1':'dst_node', 'step_1':'step',
    'amount_1':'amount', 'alert_1':'is_sar'})
right = te[['src_2','dst_2','step_2','amount_2','alert_2']].rename(columns={
    'src_2':'src_node', 'dst_2':'dst_node', 'step_2':'step',
    'amount_2':'amount', 'alert_2':'is_sar'})
event_df = (
    pd.concat([left, right], ignore_index=True)
    .drop_duplicates(subset=['src_node','dst_node','step','amount'])
    .sort_values('step').reset_index(drop=True)
)
event_df['event_id'] = np.arange(len(event_df), dtype=np.int64)
event_df['src_node'] = event_df['src_node'].astype(np.int64)
event_df['dst_node'] = event_df['dst_node'].astype(np.int64)
event_df['step']     = event_df['step'].astype(np.int32)
event_df['amount']   = event_df['amount'].astype(np.float32)
event_df['is_sar']   = event_df['is_sar'].astype(np.int8)
del left, right, te; gc.collect()
print(f'      {len(event_df):,} transactions | '
      f"steps {event_df['step'].min()}-{event_df['step'].max()} | "
      f"nodes {pd.concat([event_df['src_node'], event_df['dst_node']]).nunique():,}")

# -- Step 3: Build event indexes ----------------------------------------
print('\n[3/6] Building event indexes...')
out_index, in_index, step_index, out_steps = build_event_indexes(event_df)
del step_index; gc.collect()  # step_index unused downstream
print(f'      out_index: {len(out_index):,}  |  in_index: {len(in_index):,}')

# -- Step 4: Run all matchers -------------------------------------------
# cfg_motif.max_instances limits per-type instances in run_all_matchers.
print('\n[4/6] Running matchers...')
cfg_motif = MotifConfig()
if N_PERMS is not None:
    cfg_motif.n_permutations = N_PERMS
print(f'      delta={cfg_motif.delta}  rho=[{cfg_motif.rho_min},{cfg_motif.rho_max}]  '
      f'max_instances={cfg_motif.max_instances:,}')
all_motifs  = run_all_matchers(out_index, in_index, cfg_motif, out_steps)
raw_support = count_support(all_motifs)
print(f'      Instances: {len(all_motifs):,}  |  Support: {raw_support}')

# -- Step 5: Z-score against null model ---------------------------------
# SAMPLE_FRAC and NULL_CAP drastically reduce per-permutation RAM.
zscore_table = {}
if RUN_ZSCORE and raw_support:
    print(f'\n[5/6] Z-score ({cfg_motif.n_permutations} perms, '
          f'sample_frac={SAMPLE_FRAC}, null_cap={NULL_CAP:,})...')
    zscore_table = compute_null_zscore(
        observed_counts=raw_support, event_df=event_df, cfg=cfg_motif,
        verbose=True, sample_frac=SAMPLE_FRAC, null_cap=NULL_CAP,
    )
    print('\n  Results:')
    for mt, v in zscore_table.items():
        sig = 'OK' if v['zscore'] >= cfg_motif.z_min else '--'
        print(f"    {mt:15s}  obs={v['observed']:5d}  "
              f"mean_null={v['mean_null']:6.1f}  z={v['zscore']:6.2f}  {sig}")
else:
    print('\n[5/6] Z-score skipped.')

# -- Step 6: Filter -> features -> save ---------------------------------
print('\n[6/6] Filtering, features, saving...')
filtered = filter_motifs(
    all_motifs, cfg_motif,
    zscore_results=zscore_table if zscore_table else None,
)
print(f'      After filter: {len(filtered):,} / {len(all_motifs):,}')
print(f'      Filtered support: {count_support(filtered)}')

# Node out-degree for freq_by_degree normalisation
node_degree  = {node: len(edges) for node, edges in out_index.items()}
total_volume = float(event_df['amount'].sum())

entity_feat = build_entity_motif_features(
    filtered, zscore_table=zscore_table or None,
    node_degree=node_degree, total_volume=total_volume,
)
entity_wide = build_entity_feature_wide(entity_feat)
window_feat = build_window_motif_features(filtered, window_size=7)
print(f'      entity_feat(long)={entity_feat.shape}  '
      f'entity_feat(wide)={entity_wide.shape}  window={window_feat.shape}')

# Save all feature tables to Drive
p_entity = save_features(entity_feat, 'entity_motif_features', MOTIF_OUTPUT_DIR)
p_wide   = save_features(entity_wide, 'entity_motif_wide',     MOTIF_OUTPUT_DIR)
p_window = save_features(window_feat, 'window_motif_features', MOTIF_OUTPUT_DIR)

# Save support summary as JSON for downstream reference
p_support = os.path.join(MOTIF_OUTPUT_DIR, 'motif_support.json')
with open(p_support, 'w') as _f:
    json.dump({'raw_support': raw_support,
               'filtered_support': count_support(filtered),
               'zscore_table': zscore_table}, _f, indent=2)

print('\nMotif artifacts saved:')
for lbl, pth in [('entity_feat', p_entity), ('entity_wide', p_wide),
                  ('window_feat', p_window), ('support_json', p_support)]:
    print(f'   [{lbl:14s}]  {pth}  ({os.path.getsize(pth)/1024:.1f} KB)')

# Free all large objects to recover RAM for downstream notebooks
del all_motifs, filtered, entity_feat, entity_wide, window_feat
del out_index, in_index, out_steps, node_degree, event_df
gc.collect()